# Day 21 — 분자 프로젝트 보고서 쓰기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-3/day-21.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-3/day-21.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 2 — 숫자는 파일에서 가져온다

보고서에 숫자를 손으로 옮기면 분석을 다시 돌렸을 때 본문과 표가 어긋납니다. 저장된 표에서 계산해 문장을 만듭니다.

In [ ]:
import pandas as pd

de_toy = pd.DataFrame({"gene": ["a", "b", "c", "d"],
                       "log2FC": [-0.9, 0.2, -0.4, 0.7],
                       "p_value": [0.001, 0.60, 0.03, 0.04],
                       "FDR": [0.004, 0.60, 0.053, 0.053]})

n = len(de_toy)
n_p = int((de_toy["p_value"] < 0.05).sum())
n_fdr = int((de_toy["FDR"] < 0.05).sum())
print(f"검정한 {n}개 유전자 중 보정 전 p < 0.05는 {n_p}개, FDR < 0.05는 {n_fdr}개였다.")

### 예시 3 — 보고서에 빠진 절이 없는지 코드로 확인하기

In [ ]:
draft = """# 제목
## 질문
## 데이터
## 방법
## 결과
## 해석
"""
required = ["질문", "데이터", "QC", "방법", "결과", "해석", "한계", "인용"]
missing = [h for h in required if f"## {h}" not in draft]
print("빠진 절:", missing)

## 직접 해보기

1. `02_osdr_qc_and_pca.ipynb`의 맨 아래에 "Day 21" 셀을 만들고, 이번 주에 저장한 파일에서 보고서에 쓸 숫자를 한꺼번에 뽑습니다.

In [ ]:
# 이 셀은 Day 15–20의 산출물 파일이 필요합니다. Colab은 노트북마다 런타임이 따로라서,
# 그 파일을 이 런타임에 업로드하거나 Google Drive에 두고 경로를 맞추세요.
from pathlib import Path
import pandas as pd

MODE = "practice"          # 코드 확인 후 "real"로 바꾸세요
RAW_DIR = Path("../data/osdr_536_practice" if MODE == "practice" else "../data/osdr_536")
OUT_DIR = RAW_DIR / "processed"
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(exist_ok=True)

samples = pd.read_csv(OUT_DIR / "samples.csv")
sample_qc = pd.read_csv(OUT_DIR / "sample_qc.csv")
gene_qc = pd.read_csv(OUT_DIR / "gene_qc.csv", index_col=0)
de = pd.read_csv(OUT_DIR / "de_table.csv")
pathway = pd.read_csv(OUT_DIR / "pathway_table.csv")

facts = {
    "n_FLT": int((samples["group"] == "FLT").sum()),
    "n_GC": int((samples["group"] == "GC").sum()),
    "genes_measured": len(gene_qc),
    "genes_tested": len(de),
    "size_factor_FLT": sample_qc.loc[sample_qc["group"] == "FLT", "size_factor"].agg(["min", "max"]).tolist(),
    "size_factor_GC": sample_qc.loc[sample_qc["group"] == "GC", "size_factor"].agg(["min", "max"]).tolist(),
    "p_lt_0.05": int((de["p_value"] < 0.05).sum()),
    "expected_by_chance": round(len(de) * 0.05, 1),
    "FDR_lt_0.05": int((de["FDR"] < 0.05).sum()),
    "down_in_FLT": int(((de["p_value"] < 0.05) & (de["log2FC"] < 0)).sum()),
    "up_in_FLT": int(((de["p_value"] < 0.05) & (de["log2FC"] > 0)).sum()),
    "top5_genes": de.sort_values("p_value")["gene"].head(5).tolist(),
    "gene_sets_FDR_lt_0.05": int((pathway["FDR"] < 0.05).sum()),
}
for k, v in facts.items():
    print(f"{k}: {v}")

2. 보고서 뼈대 파일을 만듭니다. 이미 파일이 있으면 덮어쓰지 않습니다.

In [ ]:
name = "osdr_neuroinflammation_report.md" if MODE == "real" else "osdr_practice_report_skeleton.md"
report_path = REPORT_DIR / name

skeleton = f"""# OSD-536 마우스 뇌 신경염증 패널 탐색 보고서

작성자: [이름] · 작성일: [YYYY-MM-DD] · 분석 노트북: 02_osdr_qc_and_pca.ipynb, 03_osdr_pathway_analysis.ipynb

## 질문
[한두 문장. 예: ISS 비행군과 지상 대조군의 뇌 조직에서 신경염증·산화스트레스·신경기능 관련 유전자의 mRNA 수준이 다른가?]

## 데이터
[data/README.md에서 옮겨 적기: 출처, DOI, 다운로드 날짜, assay, 군별 n = FLT {facts['n_FLT']} / GC {facts['n_GC']}, 측정 유전자 {facts['genes_measured']}개]

## QC
[sample ID 연결 확인 결과, 정규화 계수 범위(FLT {facts['size_factor_FLT']}, GC {facts['size_factor_GC']}), 군과 겹치는 기술적 차이, outlier 후보와 결정, 검정에 남긴 유전자 {facts['genes_tested']}개]

## 방법
[정규화, 검출 기준, contrast(FLT − GC), 검정, 다중검정 보정, pathway 분석의 배경과 후보 기준, gene set 출처]

## 논문 분석과의 차이
[Day 18 비교표에서 옮겨 적기: 원 논문의 정규화(Positive 대조 + housekeeping), 검정(ANOVA + Tukey), 보정 여부(적혀 있지 않음), pathway 방법과 이 분석의 차이. "적혀 있지 않음"은 그대로 적고 추측해 채우지 않는다. 결과가 논문의 54개 DEG와 다른 이유를 방법 차이로 설명]

## 결과
[보정 전 p < 0.05: {facts['p_lt_0.05']}개 (우연 기대치 약 {facts['expected_by_chance']}개), FDR < 0.05: {facts['FDR_lt_0.05']}개. 방향: 감소 {facts['down_in_FLT']} / 증가 {facts['up_in_FLT']}. 후보 유전자 그림과 pathway 표에서 관찰한 것. 그림 2–3장 삽입]

## 해석
[관찰이 시사하는 것. 동사 강도 주의. 마지막에 권장 결론 형식의 두 문장]

## 한계
[표본 수, assay 범위, 기술적 요인, 방법의 단순함, 설계 조건, 일반화]

## 인용
[데이터셋 인용, 논문 인용, gene set 출처, 사용한 소프트웨어]
"""

if report_path.exists():
    print("이미 있음, 덮어쓰지 않음:", report_path)
else:
    report_path.write_text(skeleton, encoding="utf-8")
    print("뼈대 생성:", report_path)

3. `MODE = "real"`로 바꿔 1–2번을 다시 실행하고, 만들어진 `reports/osdr_neuroinflammation_report.md`를 엽니다. 대괄호를 하나씩 여러분의 문장으로 바꿉니다. 권장 순서는 데이터 → QC → 방법 → 결과 → 한계 → 해석 → 질문입니다. 사실을 옮기는 절부터 쓰고, 판단이 필요한 절은 나중에 씁니다.
4. 그림을 두세 장 넣습니다. 마크다운에서는 `![PCA](../figures/osdr_real_pca.png)`처럼 상대 경로로 넣고, 그림마다 Day 19에서 쓴 것 같은 캡션을 답니다.
5. 결과 절의 모든 숫자가 1번 셀의 출력과 일치하는지 대조합니다. p-value 기준을 쓴 문장마다 "보정 전"인지 "FDR"인지 적혀 있는지 확인합니다.
6. 해석 절의 마지막을 권장 결론 형식으로 마무리하되, "특정" 자리에 실제로 관찰한 내용을 넣습니다.
7. 아래 셀로 여덟 절이 모두 있는지, 채우지 않은 대괄호가 남았는지 확인합니다.

In [ ]:
text = report_path.read_text(encoding="utf-8")
required = ["질문", "데이터", "QC", "방법", "결과", "해석", "한계", "인용"]
missing = [h for h in required if f"## {h}" not in text]
print("빠진 절:", missing)
print("남은 대괄호 자리표시자:", text.count("["))
print("글자 수(공백 포함):", len(text))